<a href="https://colab.research.google.com/github/atikhasan007/DeepLearning/blob/main/pytorch/08_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1) Design model (input, output size , forward pass)
# 2) Construct loss and optimizer
# 3) Training loop
# - forward pass: compute prediction and loss
# - backward pass : gradients
# - update weights


In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt



**step 0: prepare data**

In [3]:
bc = datasets.load_breast_cancer()
X, y = bc.data , bc.target


In [8]:
print(X.shape)
print(y.shape)

(569, 30)
(569,)


In [10]:
n_samples, n_features = X.shape

In [11]:
n_samples, n_features

(569, 30)

In [12]:
n_samples

569

In [13]:
n_features

30

In [14]:
# train test split

X_train , x_test , Y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [15]:
# Scale
sc = StandardScaler()
X_train_Scaled = sc.fit_transform(X_train)
x_test_scaled = sc.transform(x_test)


In [16]:
X_train_Scaled_tensor = torch.from_numpy(X_train_Scaled.astype(np.float32))
x_test_scaled_tensor = torch.from_numpy(x_test_scaled.astype(np.float32))


Y_train_tensor = torch.from_numpy(Y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))


In [17]:
Y_train_tensor = Y_train_tensor.view(Y_train_tensor.shape[0], 1)
y_test_tensor = y_test_tensor.view(y_test_tensor.shape[0], 1)

In [19]:
Y_train_tensor.shape

torch.Size([455, 1])

In [20]:
y_test_tensor.shape

torch.Size([114, 1])

**step 1: Model**

In [26]:
# model
# f(x) = sigmoid(wx + b) this model work , sigmoid and the end
# custom model

class LogisticRegression(nn.Module):

  def __init__(self, n_input_features):
    super(LogisticRegression, self).__init__() # nn.Module এর constructor (init) আগে চালাও
    self.linear = nn.Linear(n_input_features, 1) # input and output size

  def forward(self, x):
    y_predicted = torch.sigmoid(self.linear(x))
    return y_predicted


model = LogisticRegression(n_features)  # model is object




**step 2: loss and optimizer**

In [27]:
learning_rate = 0.01
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(),
                            lr=learning_rate)

**step 3: training loop**

In [28]:
num_epochs = 100
for epoch in range(num_epochs):
  # forward pass
  y_predicted = model(X_train_Scaled_tensor)
  loss = criterion(y_predicted, Y_train_tensor)

  # backword pass
  loss.backward()

  # update
  optimizer.step()

  # zero gradients
  optimizer.zero_grad()

  if (epoch+1) % 10 == 0:
    print(f'epoch: {epoch+1}, loss = {loss.item():.4f}')


epoch: 10, loss = 0.5824
epoch: 20, loss = 0.4811
epoch: 30, loss = 0.4143
epoch: 40, loss = 0.3678
epoch: 50, loss = 0.3340
epoch: 60, loss = 0.3083
epoch: 70, loss = 0.2882
epoch: 80, loss = 0.2719
epoch: 90, loss = 0.2585
epoch: 100, loss = 0.2472


In [29]:
with torch.no_grad():
  y_predicted = model(x_test_scaled_tensor)
  y_predicted_cls = y_predicted.round()
  acc = y_predicted_cls.eq(y_test_tensor).sum() / float(y_test_tensor.shape[0])
  print(f'accuracy = {acc:.4f}')

accuracy = 0.9737
